# C2: Decision Tree Baseline & Readable Shallow Tree

This notebook covers **Package C — Step C2**:
1. **Baseline Decision Tree** (`DecisionTreeClassifier(random_state=SEED)`) with `scale=False` — compare **Train F1** vs **5-fold CV F1** to observe overfitting.
2. **Readable Shallow Tree** (`max_depth=4`) — log its CV score, plot the tree with `plot_tree`, and print human-readable `if/else` rules with `export_text`.

In [1]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.tree import export_text, plot_tree

from src.config import ROOT
from src.data import load_train
from src.evaluate import cv_evaluate
from src.models import make_pipeline
from src.preprocessing import EXCLUDED_COLS

df = load_train()
X, y = df.drop(columns=EXCLUDED_COLS), df["label"]
print("X shape:", X.shape, "| y shape:", y.shape)

X shape: (175341, 42) | y shape: (175341,)


## 1. Baseline Decision Tree (Default Settings — Overfitting Check)
- `make_pipeline("dt", X)` automatically sets `scale=False` (numerical features stay in real units).
- `cv_evaluate` gives the honest 5-fold cross-validation F1 and logs it to `results/metrics.csv`.
- Fitting on the full training set and scoring that same training set gives **Train F1**, showing how much a default unlimited tree memorizes training rows.

In [2]:
dt_pipe = make_pipeline("dt", X)
dt_cv = cv_evaluate("dt_baseline", dt_pipe, X, y)

dt_pipe.fit(X, y)
train_f1 = f1_score(y, dt_pipe.predict(X))

print(f"Baseline DT — Train F1: {train_f1:.4f} | CV F1: {dt_cv['f1']:.4f} +/- {dt_cv['f1_std']:.4f}")
print(f"Overfitting gap (Train - CV): {train_f1 - dt_cv['f1']:.4f}")
print(f"Tree depth: {dt_pipe.named_steps['model'].get_depth()} | Leaves: {dt_pipe.named_steps['model'].get_n_leaves()}")
dt_cv

Baseline DT — Train F1: 0.9987 | CV F1: 0.9618 +/- 0.0011
Overfitting gap (Train - CV): 0.0369
Tree depth: 43 | Leaves: 5669


{'model': 'dt_baseline',
 'split': 'cv',
 'accuracy': 0.9480212958127818,
 'accuracy_std': 0.0015699418609476784,
 'precision': 0.9629278983301732,
 'precision_std': 0.0018261167726476091,
 'recall': 0.9606170546156697,
 'recall_std': 0.001129538131493561,
 'f1': 0.9617700361013949,
 'f1_std': 0.0011360734744565114,
 'fpr': 0.07882142857142857,
 'fpr_std': 0.0040245579044822545,
 'roc_auc': 0.9424868852030703,
 'roc_auc_std': 0.0021086715802749846,
 'pr_auc': 0.953077824267791,
 'pr_auc_std': 0.0017743510207895157,
 'fit_seconds': 8.914735317230225,
 'fit_seconds_std': 0.08424034314629086}

## 2. Readable Shallow Tree (`max_depth=4`)
- Limiting depth to 4 keeps at most $2^4 = 16$ leaf rules so a human can read the model directly.
- We log `dt_shallow_d4` via `cv_evaluate` for the later **C6** interpretability trade-off table, save the tree diagram to `results/figures/dt_shallow_d4.png`, and print the `if/else` rules.

In [3]:
shallow_pipe = make_pipeline("dt", X).set_params(model__max_depth=4)
shallow_cv = cv_evaluate("dt_shallow_d4", shallow_pipe, X, y)
print(f"Shallow DT (max_depth=4) — CV F1: {shallow_cv['f1']:.4f} | CV FPR: {shallow_cv['fpr']:.4f}")

shallow_pipe.fit(X, y)
prep = shallow_pipe.named_steps["prep"]
tree = shallow_pipe.named_steps["model"]
feature_names = prep.get_feature_names_out()

fig_path = ROOT / "results" / "figures" / "dt_shallow_d4.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.figure(figsize=(26, 12))
plot_tree(
    tree,
    feature_names=feature_names,
    class_names=["Normal", "Attack"],
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Readable Decision Tree (max_depth=4, scale=False)")
plt.tight_layout()
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
shallow_cv

Shallow DT (max_depth=4) — CV F1: 0.9514 | CV FPR: 0.2166


C:\Users\mayas\AppData\Local\Temp\ipykernel_21244\1719182373.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


{'model': 'dt_shallow_d4',
 'split': 'cv',
 'accuracy': 0.9304498163754623,
 'accuracy_std': 0.0006812613280833124,
 'precision': 0.9076786973341328,
 'precision_std': 0.0007019750510397124,
 'recall': 0.9994721035520941,
 'recall_std': 0.00016882774475911349,
 'f1': 0.9513662499349499,
 'f1_std': 0.0004605155132553055,
 'fpr': 0.21664285714285714,
 'fpr_std': 0.0017835701413119686,
 'roc_auc': 0.9798923132926689,
 'roc_auc_std': 0.0005310179206343572,
 'pr_auc': 0.9856529986462725,
 'pr_auc_std': 0.0003894817882978037,
 'fit_seconds': 0.9664437770843506,
 'fit_seconds_std': 0.06964644873474046}

In [4]:
rules = export_text(tree, feature_names=list(feature_names))
print(rules)

|--- num__sttl <= 61.00
|   |--- num__sinpkt <= 0.00
|   |   |--- num__smean <= 53.00
|   |   |   |--- class: 0
|   |   |--- num__smean >  53.00
|   |   |   |--- num__sbytes <= 95.00
|   |   |   |   |--- class: 1
|   |   |   |--- num__sbytes >  95.00
|   |   |   |   |--- class: 0
|   |--- num__sinpkt >  0.00
|   |   |--- class: 0
|--- num__sttl >  61.00
|   |--- num__sbytes <= 201.00
|   |   |--- num__smean <= 54.50
|   |   |   |--- num__sloss <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- num__sloss >  0.50
|   |   |   |   |--- class: 0
|   |   |--- num__smean >  54.50
|   |   |   |--- cat__proto_icmp <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- cat__proto_icmp >  0.50
|   |   |   |   |--- class: 0
|   |--- num__sbytes >  201.00
|   |   |--- num__ct_srv_dst <= 2.50
|   |   |   |--- num__ct_srv_dst <= 1.50
|   |   |   |   |--- class: 1
|   |   |   |--- num__ct_srv_dst >  1.50
|   |   |   |   |--- class: 1
|   |   |--- num__ct_srv_dst >  2.50
|   |   |   |--- cat__stat